# RAW EDA

Exploratory Data Analysis on the `The big dataset of ultra-marathon running` Kaggle CSV, which is to be read straight from the Unity Catalog volume.

Checklist:
1. find out number of rows
2. find out number of columns
3. check schema if you have inferred it or specify an appropriate schema
4. descriptive summary of numerical fields
5. count nulls for each field
6. how many unique events are their?
7. how are ages distributed among the runners?
8. which countries are most represented?

In [0]:
VOLUME_PATH = "/Volumes/marathos/default/raw"

spark.sql(f"LIST '{VOLUME_PATH}/data'").display()

In [0]:
df = (
    spark.read.format("csv")
    .options(header=True, inferSchema=True, encoding="UTF-8")
    .load(f"{VOLUME_PATH}/data/TWO_CENTURIES_OF_UM_RACES.csv")
)
df.display()

## 1. find out number of rows

In [0]:
n_rows = df.count()
print(f"Rows: {n_rows:,}")

## 2. find out number of columns

In [0]:
n_cols = len(df.columns)
print(f"Columns: {n_cols}")
print(df.columns)

## 3. check schema if you have inferred it or specify an appropriate schema

`inferSchema=True` similar input for the bronze table, which Spark guessed the types for.

In [0]:
df.printSchema()

### Note on inferred types:

`Athlete year of birth` = `double` rather than `integer` even though years are whole numbers. Could be explained in the CSV, since it has nulls in that column, while Spark's `IntegerType` cannot hold `null` during schema, therefore, it falls back to `DoubleType` which supports `NaN`/`null`.



## 4. descriptive summary of numerical fields

In [0]:
from pyspark.sql.types import NumericType

numeric_cols = [
    f.name for f in df.schema.fields if isinstance(f.dataType, NumericType)
]
print("Numeric columns:", numeric_cols)
df.select(*numeric_cols).describe().display()

## 5. count nulls for each field

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

null_counts = df.select(
    [spark_sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]
).collect()[0].asDict()

for c, n in null_counts.items():
    print(f"{c:30s} {n:>12,} nulls  ({n / n_rows * 100:5.2f}%)")

## 6. how many unique events are their?
Note: 'Event name' often included the year, distinct 'Event name' gives 'event instances' rather than unique recurring events.

In [0]:
n_distinct_events = df.select("Event name").distinct().count()
print(f"Distinct 'Event name' values: {n_distinct_events:,}")

In [0]:
df.groupBy("Event name").count().orderBy(col("count").desc()).limit(20).display()

## 7. how are ages distributed among the runners?

In [0]:
age_df = (
    df.withColumn("age_at_event", col("Year of event") - col("Athlete year of birth"))
    # filtering out broken values (age < 10 or > 100)
      .filter((col("age_at_event") >= 10) & (col("age_at_event") <= 100))
)

age_df.select("age_at_event").describe().display()

In [0]:
age_df.groupBy("age_at_event").count().orderBy("age_at_event").display()

Databricks visualization. Run in Databricks to view.

### Extra - how are events distributed over time?

The dataset spans more than two centuries. Seeing how the number of races has grown over time, for context, could be useful later on for the dashboard.

In [0]:
df.groupBy("Year of event").count().orderBy("Year of event").display()

Databricks visualization. Run in Databricks to view.

### So, how were the events distributed over time?

From the 1980's onwards, the number of events increased dramatically.

## 8. which countries are most represented?

In [0]:
df.groupBy("Athlete country").count().orderBy(col("count").desc()).limit(10).display()